# 03 - Frequency Feature Extraction

## Prerequisites
- `01_face_detection_extraction.ipynb` must be completed
- `preprocessed_faces_integrated/` must contain extracted face images

## Run Order
This notebook can run **in parallel** with `02_spatial` and `04_semantic`.

*Fastest of the 3 extractors — CPU only, no GPU needed.*

## Output
- `extracted_features_integrated/frequency/real|fake/.../frame_N.npy` (4-dim per face)

In [ ]:
import os
import numpy as np
from pathlib import Path
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

import torch
import torchvision.transforms as transforms
from PIL import Image
from scipy.stats import skew, kurtosis

print("Imports ready (CPU-only, no GPU needed)")

In [ ]:
class Config:
    BASE_DIR = Path(".")
    PREPROCESSED_DIR = BASE_DIR / "preprocessed_faces_integrated"
    FEATURES_DIR = BASE_DIR / "extracted_features_integrated"

config = Config()
(config.FEATURES_DIR / "frequency" / "real").mkdir(parents=True, exist_ok=True)
(config.FEATURES_DIR / "frequency" / "fake").mkdir(parents=True, exist_ok=True)
print("Directories ready")

In [ ]:
class FrequencyExtractor:
    """FFT-based frequency feature extractor (4-dim: mean, var, skew, kurtosis)."""

    def __init__(self):
        self.transform = transforms.Compose([
            transforms.Resize((299, 299)),
            transforms.Grayscale(),
            transforms.ToTensor(),
        ])

    def extract(self, img_path):
        img = Image.open(img_path)
        img_tensor = self.transform(img)

        f_transform = torch.fft.fft2(img_tensor)
        f_shift = torch.fft.fftshift(f_transform)
        magnitude_spectrum = torch.log(torch.abs(f_shift) + 1).numpy().flatten()

        return np.array([
            np.mean(magnitude_spectrum),
            np.var(magnitude_spectrum),
            skew(magnitude_spectrum),
            kurtosis(magnitude_spectrum)
        ], dtype=np.float32)

print("FrequencyExtractor (FFT, 4-dim) defined")

In [ ]:
def extract_single_domain(extractor, extract_fn, domain, preprocessed_dir, output_dir):
    """Extract features for one domain from all preprocessed faces.
    Skips already-extracted features for resumability."""
    image_paths = []
    for label in ["real", "fake"]:
        label_dir = preprocessed_dir / label
        if label_dir.exists():
            for video_dir in label_dir.iterdir():
                if video_dir.is_dir():
                    for img_path in video_dir.glob("*.png"):
                        image_paths.append(img_path)

    print(f"Found {len(image_paths)} face images to process")
    if not image_paths:
        print("ERROR: No images found. Run 01_face_detection_extraction.ipynb first!")
        return 0, 0, 0

    processed = 0
    skipped = 0
    errors = 0

    for img_path in tqdm(image_paths, desc=f"Extracting {domain}"):
        relative_path = img_path.relative_to(preprocessed_dir)
        feature_path = output_dir / domain / relative_path.with_suffix(".npy")

        if feature_path.exists():
            skipped += 1
            continue

        feature_path.parent.mkdir(parents=True, exist_ok=True)
        try:
            features = extract_fn(extractor, img_path)
            np.save(feature_path, features)
            processed += 1
        except Exception as e:
            errors += 1
            if errors <= 5:
                print(f"  Error on {img_path.name}: {e}")

    print(f"\n{domain.upper()} complete: Processed={processed}, Skipped={skipped}, Errors={errors}")
    return processed, skipped, errors

In [ ]:
# Initialize and run
extractor = FrequencyExtractor()

processed, skipped, errors = extract_single_domain(
    extractor,
    lambda e, p: e.extract(p),
    "frequency",
    config.PREPROCESSED_DIR,
    config.FEATURES_DIR,
)

del extractor

## Validation

In [ ]:
# ============================================================================
# QUICK VALIDATION
# ============================================================================

print(f"\n{'='*70}")
print("VALIDATION - Frequency Features")
print(f"{'='*70}")

total = 0
for label in ["real", "fake"]:
    feat_dir = config.FEATURES_DIR / "frequency" / label
    if feat_dir.exists():
        count = sum(
            sum(1 for f in os.listdir(d) if f.endswith(".npy"))
            for d in feat_dir.iterdir() if d.is_dir()
        )
        print(f"  {label}: {count} feature files")
        total += count
print(f"  TOTAL: {total} frequency features")

if total > 0:
    sample = next((config.FEATURES_DIR / "frequency").rglob("*.npy"))
    shape = np.load(sample).shape
    print(f"  Sample shape: {shape} (expected: (4,))")
    print(f"\nFrequency extraction successful!")
    print("Next: Ensure 02 and 04 are also complete, then run 05.")
else:
    print(f"\nNo frequency features found. Check logs above.")